In [3]:
import pandas as pd

TRAIN_PATH = '/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/train.csv'
TEST_PATH = '/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/test.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("=" * 50)
print(f"Train Dataset Count: {len(train_df):,} samples")
print(f"Test Dataset Count:  {len(test_df):,} samples")
print("=" * 50)
print("\nFirst 3 rows of training data:")
print(train_df.head(3))

Train Dataset Count: 120,000 samples
Test Dataset Count:  7,600 samples

First 3 rows of training data:
   Class Index                                              Title  \
0            3  Wall St. Bears Claw Back Into the Black (Reuters)   
1            3  Carlyle Looks Toward Commercial Aerospace (Reu...   
2            3    Oil and Economy Cloud Stocks' Outlook (Reuters)   

                                         Description  
0  Reuters - Short-sellers, Wall Street's dwindli...  
1  Reuters - Private investment firm Carlyle Grou...  
2  Reuters - Soaring crude prices plus worries\ab...  


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from collections import Counter
import time

# ==========================================
# 1. Dataset Paths & Execution Device
# ==========================================
TRAIN_PATH = '/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/train.csv'
TEST_PATH = '/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/test.csv'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Processing Device: {device}")

# ==========================================
# 2. Vocabulary Builder Class
# ==========================================
class Vocabulary:
    """Maps unique words to integer token IDs."""
    def __init__(self, freq_threshold=2):
        self.itos = {0: "<pad>", 1: "<unk>"}
        self.stoi = {"<pad>": 0, "<unk>": 1}
        self.freq_threshold = freq_threshold

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 2
        for sentence in sentence_list:
            for word in str(sentence).lower().split():
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        return [self.stoi.get(word, self.stoi["<unk>"]) for word in str(text).lower().split()]

# ==========================================
# 3. Custom PyTorch Dataset
# ==========================================
class AGNewsDataset(Dataset):
    def __init__(self, csv_file, vocab=None):
        self.df = pd.read_csv(csv_file)
        # Shift class labels from 1-4 to 0-3 for PyTorch CrossEntropyLoss
        self.labels = self.df['Class Index'].values - 1
        self.texts = (self.df['Title'] + " " + self.df['Description']).values

        if vocab is None:
            self.vocab = Vocabulary(freq_threshold=2)
            self.vocab.build_vocabulary(self.texts)
        else:
            self.vocab = vocab

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text_tensor = torch.tensor(self.vocab.numericalize(self.texts[idx]), dtype=torch.long)
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.long)
        return text_tensor, label_tensor

# ==========================================
# 4. Dynamic Offset Batching Function
# ==========================================
def collate_batch(batch):
    """Combines documents into a 1D tensor and records offsets."""
    label_list, text_list, offsets = [], [], [0]
    for _text, _label in batch:
        label_list.append(_label)
        text_list.append(_text)
        offsets.append(_text.size(0))

    label_list = torch.tensor(label_list, dtype=torch.long)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list = torch.cat(text_list)
    return label_list, text_list, offsets

# ==========================================
# 5. EmbeddingBag Text Classifier
# ==========================================
class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassificationModel, self).__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode="mean")
        self.fc = nn.Linear(embed_dim, num_class)

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

# ==========================================
# 6. Data Loaders & Model Initialization
# ==========================================
print("Loading Datasets and Building Vocabulary...")
train_dataset = AGNewsDataset(TRAIN_PATH)
test_dataset = AGNewsDataset(TEST_PATH, vocab=train_dataset.vocab)

VOCAB_SIZE = len(train_dataset.vocab.stoi)
EMBED_DIM = 64
NUM_CLASSES = 4
BATCH_SIZE = 128
EPOCHS = 20

print(f"Vocabulary Size: {VOCAB_SIZE:,} unique tokens")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

model = TextClassificationModel(VOCAB_SIZE, EMBED_DIM, NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# ==========================================
# 7. 20-Epoch Training Loop
# ==========================================
print("\nStarting Training for 20 Epochs...\n")

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    model.train()
    total_loss, total_acc, total_count = 0, 0, 0

    for labels, text, offsets in train_loader:
        labels, text, offsets = labels.to(device), text.to(device), offsets.to(device)
        
        optimizer.zero_grad()
        predictions = model(text, offsets)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_acc += (predictions.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    scheduler.step()
    epoch_loss = total_loss / total_count
    epoch_acc = total_acc / total_count
    elapsed = time.time() - start_time

    # Evaluate on Test Set every 5 Epochs
    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        test_acc, test_count = 0, 0
        with torch.no_grad():
            for labels, text, offsets in test_loader:
                labels, text, offsets = labels.to(device), text.to(device), offsets.to(device)
                preds = model(text, offsets)
                test_acc += (preds.argmax(1) == labels).sum().item()
                test_count += labels.size(0)
        
        print(f"Epoch {epoch:02d}/20 | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc*100:.2f}% | Test Acc: {(test_acc/test_count)*100:.2f}% | Time: {elapsed:.1f}s")
    else:
        print(f"Epoch {epoch:02d}/20 | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc*100:.2f}% | Time: {elapsed:.1f}s")

Using Processing Device: cuda
Loading Datasets and Building Vocabulary...
Vocabulary Size: 78,093 unique tokens

Starting Training for 20 Epochs...

Epoch 01/20 | Train Loss: 0.4034 | Train Acc: 86.63% | Test Acc: 91.33% | Time: 6.1s
Epoch 02/20 | Train Loss: 0.1796 | Train Acc: 94.13% | Time: 5.5s
Epoch 03/20 | Train Loss: 0.1145 | Train Acc: 96.28% | Time: 5.5s
Epoch 04/20 | Train Loss: 0.0749 | Train Acc: 97.61% | Time: 5.4s
Epoch 05/20 | Train Loss: 0.0507 | Train Acc: 98.38% | Test Acc: 90.67% | Time: 5.4s
Epoch 06/20 | Train Loss: 0.0288 | Train Acc: 99.16% | Time: 5.4s
Epoch 07/20 | Train Loss: 0.0212 | Train Acc: 99.41% | Time: 5.4s
Epoch 08/20 | Train Loss: 0.0174 | Train Acc: 99.51% | Time: 5.4s
Epoch 09/20 | Train Loss: 0.0145 | Train Acc: 99.60% | Time: 5.4s
Epoch 10/20 | Train Loss: 0.0124 | Train Acc: 99.64% | Test Acc: 89.96% | Time: 5.4s
Epoch 11/20 | Train Loss: 0.0086 | Train Acc: 99.73% | Time: 5.3s
Epoch 12/20 | Train Loss: 0.0071 | Train Acc: 99.80% | Time: 5.5s
Ep

# Prediction Code 

In [7]:
import torch

# Map integer class indices back to original string labels
ag_news_labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

def predict_custom_text(text, model, vocab, device):
    """
    Takes a raw text sentence, numericalizes it using the built vocabulary,
    generates batch offsets, and returns the predicted label along with confidence score.
    """
    model.eval()
    with torch.no_grad():
        # 1. Lowercase and numericalize raw text string
        words = str(text).lower().split()
        numericalized = [vocab.stoi.get(word, vocab.stoi["<unk>"]) for word in words]
        
        # 2. Inspect token conversion (Debug)
        decoded_tokens = [vocab.itos.get(idx, "<unk>") for idx in numericalized]
        
        # 3. Create PyTorch Tensors
        text_tensor = torch.tensor(numericalized, dtype=torch.long).to(device)
        offsets = torch.tensor([0], dtype=torch.long).to(device) # Single document offset
        
        # 4. Model Forward Pass
        output = model(text_tensor, offsets)
        
        # 5. Get Probabilities and Class Prediction
        probabilities = torch.softmax(output, dim=1)
        predicted_idx = output.argmax(1).item()
        confidence = probabilities[0][predicted_idx].item()
        
        return ag_news_labels[predicted_idx], confidence, decoded_tokens


# ==========================================
# TEST PREDICTION FUNCTION
# ==========================================

sample_texts = [
    "Babar Azam scored a brilliant century in cricket today",
    "Imran Khan political hearing scheduled at supreme court",
    "Global stock markets drop as central bank raises interest rates",
    "NASA releases high resolution images from new space telescope"
]

print("=" * 75)
print(f"{'INPUT TEXT':<40} | {'PREDICTION':<10} | {'CONFIDENCE':<10}")
print("=" * 75)

for text in sample_texts:
    predicted_label, conf, tokens = predict_custom_text(text, model, train_dataset.vocab, device)
    print(f"{text[:37] + '...':<40} | {predicted_label:<10} | {conf*100:.1f}%")

INPUT TEXT                               | PREDICTION | CONFIDENCE
Babar Azam scored a brilliant century... | Sports     | 100.0%
Imran Khan political hearing schedule... | World      | 100.0%
Global stock markets drop as central ... | Business   | 100.0%
NASA releases high resolution images ... | Sci/Tech   | 100.0%
